# Introduction to Discrete Choice Modeling with choice-learn

*Understanding and analyzing a customer base to improve an offering.*

<img src="../images/img1.png" alt="drawing" width="600"/>



Choice modeling is a powerful tool for predicting how individuals make decisions when presented with multiple alternatives. It enables businesses to understand customer preferences and make data-driven decisions regarding pricing, product features, and market positioning. Compared to traditional machine learning, choice modeling specializes in working with inputs of variable sizes. Taking into account that the set of available alternatives changes - because some are out of stock for example - is a key stake and often not allowed by classical ML models.

By analyzing how different factors influence decision-making, businesses can strategically align their offerings with customer preferences to gain a competitive edge. Discrete choice models has proven especially effective in several use-cases among which:

- **Recommending the most relevant products** to online customers based on their browsing and purchase history.
- **Determining the optimal price** for a product to capture a targeted market share.
- **Selecting the best product assortment** for supermarkets to maximize both customer satisfaction and revenue.
- **Predicting which telecom service plans** customers will choose based on data limits, pricing, and additional features

These use cases typically involve two key steps. First, a choice model is developed using past data or survey results. This model estimates the probability that a customer will choose a particular option based on factors like price, season, customer age and other relevant criteria. The second step is to leverage the model for a given objective. By analyzing the model's predictions, businesses can identify the ideal conditions needed to achieve specific objectives, such as maximizing market share or revenue.

In this article, we will introduce choice-learn, a python package for easily implementing and testing the choice model. The library is intended for people that discover choice modeling as well as seasoned researchers. We will briefly introduce the Conditional Logit, a common model used in choice modeling and see how to implement with choice-learn.

As an example, we will explore a use case where a transportation company aims to estimate the market share across various travel options - bus, car, and plane - to optimize both pricing strategies and scheduling. This analysis will help the company make data-driven decisions to improve competitiveness and meet customer demand effectively.

<ins>
Table of Content:
</ins>

- [Hands-on: Transportation market share](#hands-on-transportation-marketshare)
- [Installing Choice-Learn](#installing-choice-learn)
- [The ModeCanada dataset](#the-modecanada-choicedataset)
- [Transforming the DataFrame into a ChoiceDataset](#transforming-the-dataframe-into-a-choicedataset)
- [A Classical Choice Model: Brief Overview of the Conditional Logit](#a-classical-choice-model-brief-overview-of-the-conditional-logit)
- [Application to ModeCanada](#application-to-modecanada)
- [Let's Dig into the code](#lets-dig-into-the-code)
- [Operating the Model](#operating-the-model)
- [Conclusion](#conclusion)

## Hands-on: Transportation market share

<img src="../images/img0.png" alt="drawing" width="600"/>

Imagine you work for a bus company planning to launch a new route between two cities. As the person in charge of the business plan, you're responsible for determining pricing and setting the bus schedule. One approach could be to analyze competitors - other bus companies - and base your decisions on their strategies. But what if there are no other bus companies on this route? In that case, your competition comes from other modes of transportation: cars, trains, and planes.


Estimating how many travelers would choose the bus over other options like a plane becomes more complex in this scenario. This is where choice modeling proves invaluable.
Discrete choice models help to simulate decision-making processes when individuals are faced with various alternatives. In this context, these models can predict a traveler's preferred mode of transportation based on factors such as price, travel time, comfort, and other relevant attributes. By applying choice modeling, you can gain deeper insights into how travelers will make their decisions, allowing you to optimize your offerings and maximize your market share.

## Installing choice-learn
Choice-Learn is your go-to choice modeling library in Python. If you need any help with its installation of use I recommend that you look here. Choice-learn is available on PyPI, you can simply install it with pip. Simply run the following command in your terminal:    

In [ ]:
!pip install choice-learn

Choice-Learn makes use of common libraries such as pandas or NumPy, as well as tensorflow and tensorflow-probability. Everything should be installed with pip.

## The ModeCanada choice dataset
We will use a well-known dataset from the choice modeling literature: **ModeCanada**. This dataset focuses on transportation choices for trips between Montréal and Toronto. You can load the dataset with the following code:

In [ ]:
from choice_learn.datasets import load_modecanada

dataset = load_modecanada(as_frame=True)
print(dataset.head())

The ModeCanada dataset contains detailed information about various travel choices made by individuals. Looking at the first sample, we can examine the data more closely:

In [ ]:
print(dataset.loc[dataset.case == 1])

Filtering for a value of a case lets us see one choice made by a user. Each row represents one alternative that was available and its characteristics.

We observe the following: the traveler had the option to choose between a car and a train and ultimately decided to drive. The dataset also provides data about the traveler's income and several attributes of the available alternatives, including cost, frequency, and travel time - both in-vehicle time (IVT) and out-of-vehicle time (OVT).

Take some time to explore the dataset to be sure you understand it well.

## Transforming the DataFrame into a ChoiceDataset

In order to work with choice-learn you need to wrap your data within a ChoiceDataset. This ChoiceDataset defines three type of information that need to be fed to a choice model:
- **The choices:** which alternative has been chosen by the user
- **The set of available options:** which alternatives were available at the time of the choice
- **Characteristics or features** that influence the user choice.

Moreover, the features are separated into two different types:
- **The 'items_features'** that describe each alternative. Common items features are the price, comfort or duration of the alternatives.
- **The 'shared_features'** that describe the context of the choice. Those are usually customer features (income, age, etc…) or describe the context. In this case, it can be information about the season or the weather for example. These features have a direct impact on the choice but are not alternative-specific.

In order to instantiate a ChoiceDataset from our DataFrame, we need to specify which column corresponds to which information:

In [ ]:
from choice_learn.data import ChoiceDataset

choice_dataset = ChoiceDataset.from_single_long_df(
  df=dataset,
  shared_features_columns=["income"],
  items_features_columns=["cost", "ivt", "ovt", "freq"],
  items_id_column="alt",
  choices_id_column="case",
  choices_column="choice",
  choice_format="one_zero",
)

- the arguments *shared_features_columns* and items_features_column specify each features type
- *choices_column* specifies which column indicates the customer's final choice
- *choice_format* specifies how the information is encoded in choice_column. Here "one_zero" is a format where 1 is given when the alternative is chosen and 0 is given otherwise
- *items_id_column* indicates which column specifies which alternative - or item - is represented in each row
- finally, *choices_id_column* indicates which column identifies each choice. This indicator has to let us regroup the rows with all the available alternatives for each choice

If you want to transform your own DataFrame to a ChoiceDataset you can either set it in the long format as our example, or you can check the package documentation to see the other formats supported (such as the wide format).

## A Classical Choice Model: Brief Overview of the Conditional Logit

Choice modeling involves understanding how individuals make decisions when presented with a set of alternatives. Each alternative, i, is associated with a utility value U(i), which depends on its attributes such as price, quality, and other relevant factors. The individual ultimately selects the alternative with the highest utility. The central task in choice modeling is to determine the utility function, U(i).

One of the foundational models in this field is the Conditional Logit Model, which assumes that utility is linearly related to the attributes of the alternatives. If we denote the characteristics of alternative *i* by $x_{i, j}$ then the utility can be expressed as:

$$ U(i) = \sum_j \beta_j x_{i, j}$$

Note that Unlike traditional regression models, the coefficients $\beta_j$​ in choice modeling may depend on the alternative being considered.


The Conditional Logit model also accounts for randomness in the decision-making process. The probability of choosing alternative *i* is modeled using the softmax function of the utilities, which introduces probabilistic behavior into the choice:

$$
\mathbb{P}(i) = \frac{e^{U(i)}}{\sum_k e^{U(k)}}
$$

This approach captures the idea that while individuals are more likely to choose options with higher utility, their decisions may still involve some uncertainty.

## Application to ModeCanada

Choosing the right choice model is often where the major questions arise. It generally boils down to two considerations: What makes logical sense? and What do I need from the model?

Let's walk through an example to break it down. We write U(c, i) the utility function that a customer *c* attributes to a transport alternative *i*.

$$
U(c, i) = \beta_i^{inter} + \beta^{price} \cdot price(i) + \beta^{freq} \cdot freq(i) + \beta^{ovt} \cdot ovt(i) + \beta_i^{income} \cdot income(c) + \beta_i^{ivt} \cdot ivt(i)
$$

- $\beta^{price}$ represents the price elasticity, reflecting how much utility decreases as the price of an alternative increases.
- Similarly to $\beta^{price}$, $\beta^{freq}$ and $\beta^{ovt}$ capture the effects of frequency and out-of-vehicle time, respectively.
- $\beta^{income}_i$ and $\beta^{ivt}_i$ are slightly different. The subscript *i* indicates that we estimate a separate coefficient for each alternative. For example, a traveler's income may affect the utility of a luxury mode of transportation differently than a simpler one.
- Finally, $\beta^{inter}_i$ is an alternative-specific constant, which represents the baseline preference for a given alternative when all other attributes are equal.

When learning one coefficient by alternative we usually select an item k and set $\beta^{coeff}_k$ = 0. It ensures the unicity of the solution. In our case, we choose $\beta^{inter}_0$ = 0 and $\beta^{income}_0$ = 0. In the following code it explains why the index 0 is absent twice.

## Let’s dig into the code

Defining such a model with choice-learn is pretty easy. Here is a code snippet that shows it all.

In [ ]:
from choice_learn.models import ConditionalLogit

# Initialize the model
model = ConditionalLogit()

# Create the different weights:
# Shared coefficients apply to all items in the list
# (e.g., price, frequency, and ovt)
model.add_shared_coefficient(feature_name="cost",
  items_indexes=[0, 1, 2, 3])

model.add_shared_coefficient(feature_name="freq",
  coefficient_name="beta_frequence",
  items_indexes=[0, 1, 2, 3])

model.add_shared_coefficient(feature_name="ovt",
  items_indexes=[0, 1, 2, 3])

# 'ivt' has a separate coefficient for each item
model.add_coefficients(feature_name="ivt",
  items_indexes=[0, 1, 2, 3])

# Add intercept and income coefficients,
# applied to all items except the first one (which is zeroed)
model.add_coefficients(feature_name="intercept",
  items_indexes=[1, 2, 3])

model.add_coefficients(feature_name="income",
  items_indexes=[1, 2, 3])

When we define a coefficient, we need to specify a feature_name that must match with the column of our DataFrame. We can either use 
*model.add_shared_coefficient*, meaning that for this feature the learned weight will be the same for all the alternatives. Or we can use 
*model.add_coefficient* and specify some items_indexes. In this case we will learn a coefficient for each of the alternatives that has its index in items_indexes.

Finally, to compute and display the model’s results, simply call the *.fit* method:

In [ ]:
history = model.fit(choice_dataset, get_report=True, verbose=2)

If you set the parameter get_report to True, you can access a summary of the estimated coefficients with *model.report*:

In [ ]:
model.report

## Operating the model

With the computed model, we can observe how market shares for each transportation mode shift as factors like price or customer income vary. In this example, we see that, as expected, when the price of an option increases, the likelihood of it being chosen decreases. Additionally, wealthier customers tend to opt for more expensive and luxurious modes of transportation, such as planes or premium train services, over budget options like buses.

Companies can use these insights to fine-tune their pricing strategies and optimize their offerings. For instance, they might introduce dynamic pricing, offering discounts during off-peak periods to capture more price-sensitive customers, or enhance premium services to attract high-income travelers.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

shared_features_by_choice = [choice_dataset.shared_features_by_choice[0][2237]]*10
items_features_by_choice = [choice_dataset.items_features_by_choice[0][2237].copy() for _ in range(10)]

colors = ["#9ae1e2", "#332851", "#ca3074", "#f6c677"]
values = []
for i in range(10):
    items_features_by_choice[i][1][0] = 20 * i
    values.append(items_features_by_choice[i][1][0])
    items_features_by_choice[i][0, 0] += 200
play_dataset = ChoiceDataset(
    shared_features_by_choice = shared_features_by_choice,
    items_features_by_choice = items_features_by_choice,
    choices = choice_dataset.choices[:10]
)

probs = model.predict_probas(play_dataset)

items = ['bus', "car", "train"]
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.fill_between(values, [0 for _ in values], probs[:, 0], label=f"air", color=colors[0])
for i in range(1, 4):
    plt.fill_between(values, np.sum(probs[:, :i], axis=1), np.sum(probs[:, :i+1], axis=1), label=f"{items[i-1]}", color=colors[i])
plt.xlabel("Bus Ticket Price")
plt.ylabel("Probability of choice")
plt.legend()

values = []
shared_features_by_choice = [choice_dataset.shared_features_by_choice[0][837].copy() for _ in range(10)]
items_features_by_choice = [choice_dataset.items_features_by_choice[0][837].copy() for _ in range(10)]
for i in range(10):
    shared_features_by_choice[i][0] += 10 * i
    values.append(shared_features_by_choice[i][0])
play_dataset = ChoiceDataset(
    shared_features_by_choice = shared_features_by_choice,
    items_features_by_choice = items_features_by_choice,
    choices = choice_dataset.choices[:10]
)

probs = model.predict_probas(play_dataset)

plt.subplot(1, 2, 2)
plt.fill_between(values, [0 for _ in values], probs[:, 0], label=f"air", color=colors[0])
for i in range(1, 4):
    plt.fill_between(values, np.sum(probs[:, :i], axis=1), np.sum(probs[:, :i+1], axis=1), label=f"{items[i-1]}", color=colors[i])
    
plt.xlabel("Customer Income")
plt.ylabel("Probability of choice")
plt.legend()


## Conclusion
This introduction to choice modeling only presented the Conditional Logit. More complex choice models exist such as the Nested Logit that takes into account substitutability between subsets of alternatives or RUMnet which is a neural-network-based choice model.

You can check the GitHub repository of Choice-Learn. The documentation also contains detailed examples of different models as well as use cases such as pricing or assortment optimization.